# 🏥 RAG Workshop - Health Check

This notebook validates your Azure environment and tests connectivity to all required services.

## Prerequisites

Before running this notebook:
1. Complete `setup.ipynb` first
2. Ensure your `.env` file exists in the project root

## What This Notebook Tests

| Service | Test |
|---------|------|
| Azure OpenAI | GPT-4.1, GPT-4.1-mini, Embeddings |
| Azure AI Search | Service connectivity |
| Document Intelligence | Layout model access |
| Content Understanding | GA API endpoint |
| Azure Storage | Blob access |

---

**Estimated time**: ~5 minutes

## Setup: Load Environment

First, let's load your configuration from the `.env` file.

In [ ]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# Add src to path for imports
project_root = Path("../../").resolve()
sys.path.insert(0, str(project_root))

print("📁 RAG Workshop Health Check")
print("=" * 50)

# Load environment variables
env_path = project_root / ".env"
if env_path.exists():
    load_dotenv(env_path)
    print(f"✅ Loaded environment from: {env_path}")
else:
    print(f"❌ .env file not found at: {env_path}")
    print("\n💡 Run setup.ipynb first to configure your environment.")
    raise FileNotFoundError(".env file not found. Run setup.ipynb first.")

print(f"\n📋 Project Root: {project_root}")
print(f"   Python: {sys.version.split()[0]}")
print(f"   Region: {os.getenv('AZURE_LOCATION', 'Not set')}")

## Test 1: Environment Variables

Verify all required environment variables are set.

In [ ]:
print("🔍 Test 1: Environment Variables")
print("=" * 50)

# Define all required variables by service
required_vars = {
    "Azure OpenAI": [
        "AZURE_OPENAI_ENDPOINT",
        "AZURE_OPENAI_API_KEY",
        "AZURE_OPENAI_API_VERSION",
        "AZURE_OPENAI_DEPLOYMENT_GPT41",
        "AZURE_OPENAI_DEPLOYMENT_GPT41_MINI",
        "AZURE_OPENAI_DEPLOYMENT_EMBEDDING",
    ],
    "Azure AI Search": [
        "AZURE_SEARCH_ENDPOINT",
        "AZURE_SEARCH_API_KEY",
    ],
    "Document Intelligence": [
        "AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT",
        "AZURE_DOCUMENT_INTELLIGENCE_KEY",
    ],
    "Content Understanding": [
        "AZURE_CONTENT_UNDERSTANDING_ENDPOINT",
        "AZURE_CONTENT_UNDERSTANDING_KEY",
        "AZURE_CONTENT_UNDERSTANDING_API_VERSION",
    ],
    "Azure Storage": [
        "AZURE_STORAGE_CONNECTION_STRING",
    ],
}

all_passed = True
env_results = {}

for service, vars_list in required_vars.items():
    print(f"\n📌 {service}:")
    service_passed = True
    for var in vars_list:
        value = os.getenv(var)
        if value:
            # Mask sensitive values
            if 'KEY' in var or 'SECRET' in var or 'CONNECTION' in var:
                display = value[:8] + "..." + value[-4:] if len(value) > 12 else "***"
            else:
                display = value[:40] + "..." if len(value) > 40 else value
            print(f"   ✅ {var}: {display}")
        else:
            print(f"   ❌ {var}: NOT SET")
            service_passed = False
            all_passed = False
    env_results[service] = service_passed

print("\n" + "=" * 50)
if all_passed:
    print("✅ Test 1 PASSED: All environment variables are set")
else:
    print("❌ Test 1 FAILED: Some environment variables are missing")

## Test 2: Azure OpenAI

Test connectivity to Azure OpenAI and verify all model deployments.

In [ ]:
from openai import AzureOpenAI

print("🔍 Test 2: Azure OpenAI")
print("=" * 50)

openai_results = {}

try:
    client = AzureOpenAI(
        azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
        api_key=os.getenv("AZURE_OPENAI_API_KEY"),
        api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2024-08-01-preview")
    )
    print(f"   Endpoint: {os.getenv('AZURE_OPENAI_ENDPOINT')}")
    print(f"   API Version: {os.getenv('AZURE_OPENAI_API_VERSION')}")
except Exception as e:
    print(f"❌ Failed to create Azure OpenAI client: {e}")
    client = None

# Test GPT-4.1
print(f"\n📌 Testing GPT-4.1 ({os.getenv('AZURE_OPENAI_DEPLOYMENT_GPT41', 'gpt-4.1')})...")
if client:
    try:
        response = client.chat.completions.create(
            model=os.getenv("AZURE_OPENAI_DEPLOYMENT_GPT41", "gpt-4.1"),
            messages=[{"role": "user", "content": "Reply with only: OK"}],
            max_tokens=10
        )
        print(f"   ✅ GPT-4.1 working - Response: {response.choices[0].message.content}")
        openai_results['gpt-4.1'] = True
    except Exception as e:
        print(f"   ❌ GPT-4.1 failed: {e}")
        openai_results['gpt-4.1'] = False

# Test GPT-4.1-mini
print(f"\n📌 Testing GPT-4.1-mini ({os.getenv('AZURE_OPENAI_DEPLOYMENT_GPT41_MINI', 'gpt-4.1-mini')})...")
if client:
    try:
        response = client.chat.completions.create(
            model=os.getenv("AZURE_OPENAI_DEPLOYMENT_GPT41_MINI", "gpt-4.1-mini"),
            messages=[{"role": "user", "content": "Reply with only: OK"}],
            max_tokens=10
        )
        print(f"   ✅ GPT-4.1-mini working - Response: {response.choices[0].message.content}")
        openai_results['gpt-4.1-mini'] = True
    except Exception as e:
        print(f"   ❌ GPT-4.1-mini failed: {e}")
        openai_results['gpt-4.1-mini'] = False

# Test Embeddings
print(f"\n📌 Testing Embeddings ({os.getenv('AZURE_OPENAI_DEPLOYMENT_EMBEDDING', 'text-embedding-3-large')})...")
if client:
    try:
        response = client.embeddings.create(
            model=os.getenv("AZURE_OPENAI_DEPLOYMENT_EMBEDDING", "text-embedding-3-large"),
            input="RAG Workshop test"
        )
        embedding_dim = len(response.data[0].embedding)
        print(f"   ✅ Embeddings working - Dimension: {embedding_dim}")
        openai_results['embedding'] = True
    except Exception as e:
        print(f"   ❌ Embeddings failed: {e}")
        openai_results['embedding'] = False

print("\n" + "=" * 50)
if all(openai_results.values()):
    print("✅ Test 2 PASSED: All Azure OpenAI models are working")
else:
    print("❌ Test 2 FAILED: Some Azure OpenAI models are not working")

## Test 3: Azure AI Search

Test connectivity to Azure AI Search service.

In [ ]:
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.core.credentials import AzureKeyCredential

print("🔍 Test 3: Azure AI Search")
print("=" * 50)

search_passed = False

try:
    endpoint = os.getenv("AZURE_SEARCH_ENDPOINT")
    api_key = os.getenv("AZURE_SEARCH_API_KEY")
    
    print(f"   Endpoint: {endpoint}")
    
    # Create index client
    index_client = SearchIndexClient(
        endpoint=endpoint,
        credential=AzureKeyCredential(api_key)
    )
    
    # List existing indexes (tests connectivity)
    indexes = list(index_client.list_indexes())
    print(f"\n   ✅ Connection successful!")
    print(f"   📋 Existing indexes: {len(indexes)}")
    for idx in indexes:
        print(f"      • {idx.name}")
    
    search_passed = True

except Exception as e:
    print(f"   ❌ Connection failed: {e}")

print("\n" + "=" * 50)
if search_passed:
    print("✅ Test 3 PASSED: Azure AI Search is accessible")
else:
    print("❌ Test 3 FAILED: Cannot connect to Azure AI Search")

## Test 4: Azure AI Document Intelligence

Test connectivity to Document Intelligence service.

In [ ]:
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.core.credentials import AzureKeyCredential

print("🔍 Test 4: Azure AI Document Intelligence")
print("=" * 50)

di_passed = False

try:
    endpoint = os.getenv("AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT")
    api_key = os.getenv("AZURE_DOCUMENT_INTELLIGENCE_KEY")
    
    print(f"   Endpoint: {endpoint}")
    
    # Create client
    di_client = DocumentIntelligenceClient(
        endpoint=endpoint,
        credential=AzureKeyCredential(api_key)
    )
    
    # Check available models (tests connectivity)
    print(f"\n   ✅ Connection successful!")
    print(f"   📋 Available models for workshop:")
    print(f"      • prebuilt-layout (for structured document extraction)")
    print(f"      • prebuilt-read (for basic OCR)")
    print(f"      • prebuilt-document (general document analysis)")
    
    di_passed = True

except Exception as e:
    print(f"   ❌ Connection failed: {e}")
    print(f"\n💡 Troubleshooting:")
    print(f"   1. Verify the endpoint URL ends with cognitiveservices.azure.com")
    print(f"   2. Check that the API key is valid")
    print(f"   3. Ensure the AI Services resource is deployed")

print("\n" + "=" * 50)
if di_passed:
    print("✅ Test 4 PASSED: Document Intelligence is accessible")
else:
    print("❌ Test 4 FAILED: Cannot connect to Document Intelligence")

## Test 5: Azure AI Content Understanding

Test connectivity to Content Understanding service (GA API).

In [ ]:
import requests

print("🔍 Test 5: Azure AI Content Understanding")
print("=" * 50)

cu_passed = False

try:
    endpoint = os.getenv("AZURE_CONTENT_UNDERSTANDING_ENDPOINT")
    api_key = os.getenv("AZURE_CONTENT_UNDERSTANDING_KEY")
    api_version = os.getenv("AZURE_CONTENT_UNDERSTANDING_API_VERSION", "2025-11-01")
    
    print(f"   Endpoint: {endpoint}")
    print(f"   API Version: {api_version}")
    
    # Test endpoint availability by listing analyzers
    list_url = f"{endpoint.rstrip('/')}/contentunderstanding/analyzers?api-version={api_version}"
    
    headers = {
        "Ocp-Apim-Subscription-Key": api_key,
        "Content-Type": "application/json"
    }
    
    response = requests.get(list_url, headers=headers)
    
    if response.status_code == 200:
        analyzers = response.json().get('value', [])
        print(f"\n   ✅ Content Understanding GA API accessible!")
        print(f"   📋 Existing analyzers: {len(analyzers)}")
        for analyzer in analyzers[:5]:  # Show first 5
            print(f"      • {analyzer.get('analyzerId', 'unknown')}")
        cu_passed = True
    elif response.status_code == 404:
        # Endpoint might not have any analyzers yet, but API is accessible
        print(f"\n   ✅ Content Understanding endpoint is accessible")
        print(f"   📋 No analyzers configured yet (this is normal for new deployments)")
        cu_passed = True
    else:
        print(f"\n   ❌ API returned status code: {response.status_code}")
        print(f"   Response: {response.text[:200]}")

except Exception as e:
    print(f"   ❌ Connection failed: {e}")
    print(f"\n💡 Note: Content Understanding is GA with API version 2025-11-01")
    print(f"   Available regions: westus, swedencentral, australiaeast")

print("\n" + "=" * 50)
if cu_passed:
    print("✅ Test 5 PASSED: Content Understanding is accessible")
else:
    print("❌ Test 5 FAILED: Cannot connect to Content Understanding")

## Test 6: Azure Storage

Test connectivity to Azure Storage and verify containers exist.

In [ ]:
from azure.storage.blob import BlobServiceClient

print("🔍 Test 6: Azure Storage")
print("=" * 50)

storage_passed = False

try:
    connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
    
    # Create client
    blob_service_client = BlobServiceClient.from_connection_string(connection_string)
    
    # Get account info
    account_info = blob_service_client.get_account_information()
    print(f"   ✅ Connected to storage account")
    print(f"   📋 SKU: {account_info.get('sku_name', 'Unknown')}")
    
    # List containers
    print(f"\n   📋 Containers:")
    containers = list(blob_service_client.list_containers())
    
    expected_containers = [
        os.getenv("AZURE_STORAGE_CONTAINER_DOCUMENTS", "documents"),
        os.getenv("AZURE_STORAGE_CONTAINER_FIGURES", "figures")
    ]
    
    existing_containers = [c['name'] for c in containers]
    
    for container_name in expected_containers:
        if container_name in existing_containers:
            print(f"      ✅ {container_name}")
        else:
            print(f"      ⚠️ {container_name} (not found - will be created as needed)")
    
    storage_passed = True

except Exception as e:
    print(f"   ❌ Connection failed: {e}")

print("\n" + "=" * 50)
if storage_passed:
    print("✅ Test 6 PASSED: Azure Storage is accessible")
else:
    print("❌ Test 6 FAILED: Cannot connect to Azure Storage")

## Test 7: Python Dependencies

Verify all required Python packages are installed with correct versions.

In [ ]:
import importlib.metadata

print("🔍 Test 7: Python Dependencies")
print("=" * 50)

required_packages = [
    ("azure-ai-documentintelligence", "1.0.0"),
    ("azure-search-documents", "11.4.0"),
    ("azure-identity", "1.0.0"),
    ("azure-storage-blob", "12.0.0"),
    ("openai", "2.0.0"),
    ("pandas", "2.0.0"),
    ("pillow", "10.0.0"),
    ("python-dotenv", "1.0.0"),
    ("tqdm", "4.0.0"),
]

all_installed = True

for package_name, min_version in required_packages:
    try:
        version = importlib.metadata.version(package_name)
        print(f"   ✅ {package_name}: {version}")
    except importlib.metadata.PackageNotFoundError:
        print(f"   ❌ {package_name}: NOT INSTALLED")
        all_installed = False

# Check optional packages
print(f"\n📋 Optional packages:")
optional_packages = [
    "graphrag",
    "azure-ai-projects",
    "azure-ai-agents",
]

for package_name in optional_packages:
    try:
        version = importlib.metadata.version(package_name)
        print(f"   ✅ {package_name}: {version}")
    except importlib.metadata.PackageNotFoundError:
        print(f"   ⚠️ {package_name}: Not installed (optional for later modules)")

print("\n" + "=" * 50)
if all_installed:
    print("✅ Test 7 PASSED: All required packages are installed")
else:
    print("❌ Test 7 FAILED: Some packages are missing")
    print("   Run: pip install -r requirements.txt")

## 📊 Health Check Summary

Run this cell to see a summary of all tests.

In [ ]:
print("\n" + "=" * 60)
print("📊 RAG WORKSHOP - HEALTH CHECK SUMMARY")
print("=" * 60)

# Collect results from previous tests
# These variables should be set by the test cells above
summary = {
    "Environment Variables": all(env_results.values()) if 'env_results' in dir() else False,
    "Azure OpenAI": all(openai_results.values()) if 'openai_results' in dir() else False,
    "Azure AI Search": search_passed if 'search_passed' in dir() else False,
    "Document Intelligence": di_passed if 'di_passed' in dir() else False,
    "Content Understanding": cu_passed if 'cu_passed' in dir() else False,
    "Azure Storage": storage_passed if 'storage_passed' in dir() else False,
    "Python Dependencies": all_installed if 'all_installed' in dir() else False,
}

print("\n| Test                    | Status |")
print("|-------------------------|--------|")

for test_name, passed in summary.items():
    status = "✅ PASS" if passed else "❌ FAIL"
    print(f"| {test_name:<23} | {status} |")

print("\n" + "=" * 60)

all_tests_passed = all(summary.values())

if all_tests_passed:
    print("\n🎉 ALL TESTS PASSED!")
    print("\nYour environment is fully configured for the RAG Workshop.")
    print("\n➡️ Next step: Start Module 1 - The Problem with Naive RAG")
    print("   Open: ../module-1-naive-rag/README.md")
else:
    print("\n⚠️ SOME TESTS FAILED")
    print("\nPlease review the failed tests above and:")
    print("  1. Check your .env file for correct values")
    print("  2. Verify Azure resources are deployed")
    print("  3. Run the failed test cells again after fixing")
    print("\n💡 Common issues:")
    print("  • Wrong region (use swedencentral for Content Understanding)")
    print("  • Missing model deployments (gpt-4.1, gpt-4.1-mini, text-embedding-3-large)")
    print("  • Incorrect API keys or endpoints")

---

## Next Steps

If all tests passed, you're ready to begin the workshop!

**[→ Module 1: The Problem with Naive RAG](../module-1-naive-rag/README.md)**

If tests failed, review the troubleshooting tips in each test section or re-run `setup.ipynb`.